# ABA について

ABA (Articualte-Body Algorithm)は、複数の剛体が関節で接続されたマルチボディシステムの順動力学(Forward Dynamics)を計算するアルゴリズムであり、その計算量は$O(n)$である。

RNEM(Recursive Newton-Euler Algorithm) が関節位置・速度・加速度から関節トルクを求める逆動力学であるのに対し、ABAでは関節位置・速度・トルクから関節角加速度を求める。

一般的な運動方程式では、関節空間の質量行列を用いて関節角加速度を計算するが、ABAでは Articualted-Body Inertia (ABI)を利用し、質量行列を明示的に構築することなく関節角速度を計算する。

ABIは、子リンクの未知の関節角加速度を代数的に消去することで得られる、部分機構の等価慣性である。ABAではSpatial Vectorを用いて、この等価慣性とバイアス力を末端リンクから親リンクに再帰的に伝達する。

本Noteでは、ABIの基本的な考え方、ABAの計算手順、およびSpatial Newton-Euler方程式からの数学的導出について整理する。

## 1. ABI (Articulated-Body Inertia)

### 1.1 基本的な考え方

#### 1.1.1 子リンクの動作の角加速度目標値を設定する場合

水平方向に動く2リンクのアームシステムを考える。ここでは、Link2の相対運動を拘束した場合と自由運動させた場合を比較し、Link1からみた等価慣性がどのように変化するのかを検討する。

まず、Link2にも目標とする運動を与えた場合、RNEAで導いたようにLink2単体を動作させるための関節駆動力がLink1から伝わり、その反力をLink1の関節で補償する必要があった。この運動を図で表すと以下となる。Link2のLink1に対する相対角速度、相対角加速度はゼロだが、Link1の運動によりLink2の運動 ${^2}\boldsymbol{V}_2, {^2}\boldsymbol{A}_2$ が発生する。これよりLink2は最初に位置姿勢から、約45度に傾くためにトルクを発生させる必要がある。このトルクをLink1の関節でも補償する必要がある。

<p align="center">
<img src="./images/ABI_link2_is_motion.png" style="width:40%;" />
</p>

#### 1.1.2 子リンクは自由運動すると設定する場合

Link2をアクチュエーターにより制御せず、自由運動とした場合を考える。

理想的な回転関節では、以下のように基本的には並進方向の拘束力を伝達する。

- $J_2$ は並進方向の拘束力を伝達する。
- $J_2$ の回転軸まわりのトルクは伝達しない。
- それ以外の拘束モーメントは、3D関節であれば伝達され得る

<p align="center">
<img src="./images/ABI_lini2_is_free.png" style="width:55%;" />
</p>

これにより、図では、$J_2$に水平方向と鉛直方向に$H$と$V$が生じる。

$$
F_{J_2} = \begin{bmatrix} \boldsymbol{H} \\ \boldsymbol{V} \end{bmatrix}
$$

オレンジ色矢印の拘束力がLink2には伝わり、紫色矢印が反力としてLink1に伝わる。自由運動であるため、Link2にはLink1から関節駆動力が伝達されないが、この拘束力によりLink2では $G_2$ まわりのモーメントが生じる。図では$e_{z_2}$が回転軸であるため、以下のような方程式である。

$$
I_{G_2} \ \alpha_2 = (\boldsymbol{r}_{G_2 J_2} \times F_{J_2})_z
$$

関節2を自由にした場合、関節軸まわりの拘束トルクは発生しない。一方、並進方向の拘束力は伝達される。Link2はLink1に対して相対的に角加速度を持てるため、両リンクを一体として動かす場合とことなる拘束反力が発生する。

Link1はJ2から受ける反力により、$O_1$まわりに以下のモーメント発生させる。このモーメントは図の初期姿勢のように$J_2$が$O_1$の水平方向に位置している場合である。

$$
N_{O_1,z} = \boldsymbol{r}_{O_1 J_2,x} F_{J_2 \rightarrow 1, y}
$$

よって、Link1はLink2が自由運動するとき、Link1自身を目標とする運動で動かすための慣性力と外力として$N_{O_1}$が生じる。これは、Link2の相対運動を拘束した場合と比べて、Link2単体を動作させるための関節駆動力が生じない分小さい値であるといえる。これはあくまで初期姿勢で一直線となっている状態からLink1を目標運動で動作させる場合である。

影響を簡単な運動方程式で考える。

速度依存項を除いた平面2リンクアームの運動方程式は以下のようになる。左辺は関節空間慣性行列の成分である。

$$
m_{11} \ddot{q}_1 + m_{12} \ddot{q}_2 = \tau_1 \\
m_{21} \ddot{q}_1 + m_{22} \ddot{q}_2 = \tau_2 \\
$$

Link2を自由とした場合、$\tau_2= 0$ であるため、以下となる。

$$
\ddot{q}_2 = -m_{21}/m_{22} \ \ddot{q_1}
$$

この式はLink1が加速すると、Link2が相対的に加速することを示している。これを関節1の式に代入すると、以下となる。

$$
\underbrace{(m_{11} - (m_{12}m_{21})/m_{22})}_{M_{11}^{\text{eff}}} \ddot{q}_1 = \tau_1
$$

$m_{11}$はLink1とLink2の両方の慣性モーメント、質量、重心位置などが考慮された項である。[CartPoleの運動方程式](../../CartPole-2D/notes/cartpole_2d_newton_euler.ipynb) を確認すると $m_{11}$の位置は $m_1 + m_2$ となっている。実際の2link平面アームの場合は次のような形式である。

$$
m_{11} = I_{G_1} + m_1 r_{G_1}^2 + I_{G_2} + m_2(l_1^2 + r_{G_2}^2 + 2 l_1 r_{G_2} \cos(q_2))
$$

よって、Link2を自由動作させたときにはLink1にとっての等価慣性($M_{11}^{\text{eff}}$)は、Link2を固定した場合の等価慣性$m_{11}$よりも小さくなる。

$$
M_{11}^{\text{eff}} \le  m_{11}
$$

#### 1.1.3 親リンクで子リンクの等価慣性を動かす

1.1.2節では、子リンクを制御しない劣駆動のような構成であった。そして、$\ddot{q}_2$をLink1の運動方程式に代入することで$\ddot{q_1}$にとってのLink1とLink2の等価慣性を求めた。2リンクロボットの両方の関節にトルクを与えたときにその時の動作を求める場合を考える。

ここでも同じく速度依存項を除いた平面２リンクアームの運動方程式を用いる。

$$
m_{11} \ddot{q}_1 + m_{12} \ddot{q}_2 = \tau_1 \\
m_{21} \ddot{q}_1 + m_{22} \ddot{q}_2 = \tau_2 \\
$$

まず、Link2の角加速度は第２式より以下で表現できる。これは、$\ddot{q}_1$ により $\ddot{q}_2$ が求まる形式である。 

$$
\ddot{q}_2 = m_{22}^{-1}(\tau_2 - m_{21} \ddot{q}_1)
$$

この式を第１式に代入すると、

$$
m_{11} \ddot{q}_1 + m_{12} m_{22}^{-1}(\tau_2 - m_{21} \ddot{q}_1) = \tau_1
$$

この式を整理すると次の運動方程式となる。この中で$\tau_1, \tau_2$が既知であり、$\ddot{q}_1$は未知の値であるとする。

$$
\underbrace{(m_{11} - m_{12} m_{22}^{-1} m_{21})}_{M_{11}^{\text{eff}}} \ddot{q}_1 = \tau_1 - m_{12} m_{22}^{-1} \tau_2
$$

質点の並進運動を考えた場合、右辺の制御トルクと外力の合計により、左辺の加速度の運動が慣性$m$により決まるという形式である。

$$
m a = F_{\text{control}} + \sum F_{\text{ext}}
$$

上式も同じ形式となっており、$\tau_1$ はLink1の制御トルクであり、第2項はLink2の制御トルクがLink1の運動に及ぼす影響をあらわす。Link1単体の視点では、他リンクから伝達される力・モーメントは外力として扱える。ここでは、その影響を関節2の角加速度を消去することで、関節1に対する等価的な一般化力として表現している。そして、加速度$\ddot{q}_1$ にとっての慣性は $M_{11}^{\text{eff}}$ となる。

式を変形すると次のようになるため、関節1で必要な$\tau_1$は、関節1から見た子リンクを含む等価慣性による加速に必要なトルクと、関節2の駆動$\tau_2$の影響を合わせたものとなる。

$$
\tau_1 = {M_{11}^{\text{eff}}} \ddot{q}_1 + m_{12} m_{22}^{-1} \tau_2
$$

このように、子リンクの未知の関節加速度を消去したすると、親リンクの運動に対する等価慣性を求めることが出来る。ABAではこれと同様の考え方をSpatial Vectorに拡張し、リンクのSpatial Acceleartionに対応する等価慣性をArticulated-Body Inertia(ABI) として表現する。
上記の導出は関節空間のスカラー値であるが、ABAで扱うABIは$6 \times 6$行列であり、以下のような形である。

$$
\boxed{
\boldsymbol{I}_1^A = \mathcal{I}_1 + X^T \bar{I}_2^A X
}
$$


### 1.2 Articulated-Body Inertia の定義

#### 1.2.1 Spatial Newton-Euler から考える

[RNEA](Spatial-RNEA.ipynb)で求めた、単一剛体のSpatial Newton-Eulerの方程式は以下であった。

$$
\boldsymbol{f}_i^{\text{net}} = \mathcal{I}_i \boldsymbol{A}_i + \text{crf}(\boldsymbol{V}_i)\mathcal{I}_i \boldsymbol{V}_i
$$

- $\mathcal{I}_i$ : Link i の単体のSpatial Inertia
- $\boldsymbol{A}_i$ : Link i のSpatial Acceleration
- $\boldsymbol{V}_i$ : Link i の Spatial Velocity
- $\boldsymbol{f}_i^{\text{net}}$ : Link i の運動に必要なSpatial Force (慣性項)

この式を$\boldsymbol{p}_i = \text{crf}(\boldsymbol{V}_i)\mathcal{I}_i \boldsymbol{V}_i$ とすると、以下で表現できる。$\boldsymbol{p}_i$は速度$\boldsymbol{V}_i$に依存するバイアス力である。

$$
\boldsymbol{f}_i^{\text{net}} = \mathcal{I}_i \boldsymbol{A}_i + \boldsymbol{p}_i
$$

#### 1.2.2 複数リンクの場合

今度は単体ではなく、Link i の先に複数リンク接続されている場合を考える。

<p align="center">
<img src="./images/ABI_multi_links.png" style="width:60%;" />
</p>

Link i を加速させると、その先の Link i+1, Link i+2 にも運動が生じる。<br>
ただし、各リンク間の関節は 1.1節のように自由に加速できる。

そこで、Link i とその先の部分機構全体について、力と加速度の関係を以下のように表す。

$$
\boxed{
\boldsymbol{f}_i^A = \boldsymbol{I}_i^A \boldsymbol{A}_i + \boldsymbol{p}_i^A
}
$$

それぞれの項目は以下で定義される。

- $\boldsymbol{f}_i^A$ : Link i とその子孫リンクからな部分機構に、親リンク側から作用する作用するSpatial Force
- $\boldsymbol{I}_i^A$ : Link i 以降の関節自由度を考慮した等価慣性 (Articulated-Body Inertia)
- $\boldsymbol{A}_i$ : Link i の Spatial Acceleration
- $\boldsymbol{p}_i^A$ : 部分機構全体のバイアス力 (外力はこの中に含める)

### 1.3 ABIの再帰計算の考え方

2リンクの場合で考え方を示す。

末端のLink 2 には子リンクがないため、Link 2 のABIは以下となる。

$$
\boldsymbol{I}_2^A = \mathcal{I}_2
$$

Link 1 では Link 2 の関節自由度を考慮した等価慣性を加え、以下となる。$X$はLink 1 から Link 2 へのSpatial Motion Transform である。

$$
\boldsymbol{I}_1^A = \mathcal{I}_1 + X^T \bar{I}_2^A X
$$

ここで$\bar{I}_2^A$は、関節2の加速度自由度を消去した後の等価慣性である。この式はSchur補完により導くことが出来る。

$$
\bar{I}_2^A = \boldsymbol{I}_2^A - \boldsymbol{I}_2^A \boldsymbol{S}_2 \left( \boldsymbol{S}_2^T \boldsymbol{I}_2^A \boldsymbol{S}_2 \right)^{-1} \boldsymbol{S}_2^T \boldsymbol{I}_2^A
$$

#### 1.3.1 親リンクへ伝えるもの

RNEAでは 加速度が既知であるため、Backward Recursionによって、子リンクのSpatial Force を親リンクへ伝達している。<br>
ABAでは、加速度が未知になるため、Spatial Force を加速度に依存する項と依存しない項に分け、その関係を親リンクへ伝達する。




## 2. ABAの計算の流れ


### 2.1 ABAで求めるもの

RNEAは逆動力学を計算するアルゴリズムであり、位置、速度、加速度から関節アクチュエータの動力を求める。

$$
(q, \dot{q}, \ddot{q}) \longrightarrow \tau
$$

ABAは順動力学を計算するアルゴリズムであり、位置、速度、関節アクチュエータの動力から加速度を求める。

$$
(q, \dot{q}, \tau) \longrightarrow \ddot{q}
$$

一般的な運動方程式は

$$
M(q)\ddot{q} + h(q, \dot{q}) = \tau
$$

であるため、加速度を慣性行列$M(q)$の逆行列を求めることで以下のように計算することもできる。

$$
\ddot{q}= M(q)^{-1}(\tau -h(q, \dot{q}))
$$

しかし、ABAでが$M(q)$を明示的に構築することなく、Spatial Vectorを利用した再計算によって$\ddot{q}$を求めることが出来る。<br>ABAは通常の木構造ロボットでは、関節自由度に対して線形の計算量$O(n)$で順動力学を計算することができることが特徴である。

### 2.2 ABAの3 pass

ABAの以下の3回の再帰計算のPassで関節角加速度を求める。

|パス|計算方向|主な計算項目|目的|
|:---|:---|:---|:---|
|Pass 1| Base $\rightarrow$ End | $V_i, c_i$, 初期バイアス力, 初期ABI | 速度依存項の計算, 初期値計算|
|Pass 2| End $\rightarrow$ Base | $I_i^A, p_i^A, U_i, d_i, u_i$ | 関節自由度を消去して等価慣性を集約|
|Pass 3| Base $\rightarrow$ End | $\ddot{q}_i, A_i$ | 関節加速度を求める|

以下でそれぞれのパスの式を確認する。

### 2.3 Pass 1 : Forward Recursion

Pass 1では、各リンクのSpatial Veclocity、加速度の速度依存項と初期バイアス力を計算する。

親リンクを$p(i)$とし、親座標系からリンク$i$の座標系へのSpatial Motion Transformを以下と定義する。

$$
X_i = X_{i \leftarrow p(i)}
$$

以下では各リンクのSpatial Vectorは、そのリンク座標で示すとして、左上の表記を省略する。例えば、リンク$i$座標で示したリンク
$i$のSpatial Velocity ${^i}\boldsymbol{V}_{i}$だが、これを単に$\boldsymbol{V}_{i}$と示す。

#### 2.3.1 Spatial Velocity

RNEAと同じ再帰式を利用する。

$$
\boxed{
\boldsymbol{V}_i = X_i \boldsymbol{V}_{p(i)} + \boldsymbol{S}_i \dot{q}_i
}
$$

- $\boldsymbol{V}_i$ : Link i のSpatial Velocity
- $\boldsymbol{S}_i$ : Joint i Motion Subspace
- $\dot{q}_i$ : 関節速度

#### 2.3.2 Spatial Accelerationの速度依存項

RNEAではSpatial Accelerationは以下の再帰式であった。

$$
\boldsymbol{A}_i = X_i \boldsymbol{A}_{p(i)} + \boldsymbol{S}_i \ddot{q}_i + \text{crm}(\boldsymbol{V}_i) \boldsymbol{S}_i \dot{q}_i
$$

上式の速度に依存する加速度項は以下の$\boldsymbol{c}_i$である。

$$
\boxed{
\boldsymbol{c}_i = \text{crm}(\boldsymbol{V}_i) \boldsymbol{S}_i \dot{q}_i
}
$$

これは固定軸の1自由度関節についての速度依存項である。より、一般的なMotion Subspaceが時間変化する場合には、$\dot{\boldsymbol{S}}\dot{q}_i$ に相当する項も考慮する。

Pass 1 では$\ddot{q}_i$が未知であるため、$\boldsymbol{A}_i$自体の計算は行わず、Spatial Velocityに関する項を計算する。

#### 2.3.3 初期バイアス力の設定

1.2節で説明を行った、各リンクの速度に依存するバイアス力 $\boldsymbol{p}_i^A$ を計算する。外力はこの計算に含める。

$$
\boxed{
\boldsymbol{p}_i^A = \text{crf}(\boldsymbol{V}_i) \mathcal{I}_i \boldsymbol{V}_i - \boldsymbol{f}_{\text{ext,i}}
}
$$

ここでは、各リンク単体のバイアス力を初期値として設定する。子リンクからの寄与は次のPass2で加算される。

### 2.3.4 初期ABIの設定

以下のように、各リンク単体のSpatial InertiaをABIの初期値として設定する。

$$
\boxed{
\boldsymbol{I}_i^A = \mathcal{I}_i
}
$$

### 2.4 Pass 2 : Backward Recursion

ここがABA計算の中心となる。

先端リンクからベースリンクへ向かって、関節自由度を考慮した等価慣性とバイアス力を集約する。

#### 2.4.1 関節方向への慣性の射影

$$
\boxed{\boldsymbol{U}_i = \boldsymbol{I}_i^A \boldsymbol{S}_i}, \quad \boxed{{d}_i = \boldsymbol{S}_i^T \boldsymbol{U}_i}
$$

1自由度関節なら、以下の次元となる。
- $\boldsymbol{U}_i$ : $6 \times 1$ ベクトル
- ${d}_i$ : スカラー

${d}_i$は、現在の部分機構のABIを関節$i$の運動方向へ射影した慣性に相当する。

#### 2.4.2 関節トルクからバイアス力を差し引く

$$
\boxed{
u_i = \tau_i - \boldsymbol{S}_i^T \boldsymbol{p}_i^A
}
$$

$u_i$は関節加速度の計算に利用するトルクの残差となる。

#### 2.4.3 関節自由度を消去した等価慣性の更新

等価慣性を以下の計算で更新する。先端リンクでは$\boldsymbol{I}_i^A$はPass 1 で設定した値を用い、それ以降は2.4.5で子リンクを集約した値になる。

$$
\boxed{
\bar{\boldsymbol{I}}_i^A = \boldsymbol{I}_i^A - \boldsymbol{U}_i d_i^{-1} \boldsymbol{U}_i^T
}
$$

これは1.3節で導入した以下の等価慣性と同じである。

$$
\boldsymbol{I}_i^A - \boldsymbol{I}_i^A \boldsymbol{S}_i \left(\boldsymbol{S}_i^T \boldsymbol{I}_i^A \boldsymbol{S}_i \right)^{-1} \boldsymbol{S}_i^T\boldsymbol{I}_i^A
$$

#### 2.4.4 バイアス力の更新

関節自由度を消去した後のバイアス力を以下の計算で更新する。先端リンクでは$\boldsymbol{p}_i^A$はPass 1 で設定した値を用い、
それ以降は2.4.5で子リンクを集約した値になる。

$$
\boxed{
\bar{\boldsymbol{p}}_i^A = {\boldsymbol{p}}_i^A + \bar{\boldsymbol{I}}_i^A {\boldsymbol{c}}_i + \boldsymbol{U}_i d_i^{-1} u_i
}
$$

#### 2.4.5 親リンクへの伝達

Pass 1で設定した親リンクの初期値に、更新した子リンクの値を親リンク座標系へ変換して伝達する。

$$
\boxed{
\boldsymbol{I}_{p(i)}^A \leftarrow \boldsymbol{I}_{p(i)}^A + X_i^T \bar{\boldsymbol{I}}_i^A X_i
}
$$

$$
\boxed{
\boldsymbol{p}_{p(i)}^A \leftarrow \boldsymbol{p}_{p(i)}^A + X_i^T \bar{\boldsymbol{p}}_i^A
}
$$

この後、親リンクでの計算を2.4.1から行う。

### 2.5 Pass 3 : Forward Recursion

最後に、ベースから先端リンクに向かって関節加速度を求める。

親リンクのSpatial Accelerationが既知になったとき、子リンクのSpatial Accelerationの関節角加速度を除いた項を以下とする。

$$
\boxed{
\boldsymbol{a}_i^{\text{pre}} = X_i \boldsymbol{A}_{p(i)} + \boldsymbol{c}_i
}
$$

関節角加速度は以下となる。

$$
\boxed{
\ddot{q}_i = d_i^{-1} \left( u_i - \boldsymbol{U}_i^T \boldsymbol{a}_i^{\text{pre}}\right)
}
$$

そして Spatial Accelerationは以下となる。

$$
\boxed{
\boldsymbol{A}_i = \boldsymbol{a}_i^{\text{pre}} + \boldsymbol{S}_i \ddot{q}_i
}
$$


ベース座標が固定ベースの場合、重力をベース加速度へ組み込む標準的な実装では、

$$
\boldsymbol{A}_0 = \begin{bmatrix} 0 \\ -\boldsymbol{g} \end{bmatrix}
$$

重力を外力として明示的に2.3.3節で扱い場合は、二重計上しないように注意する、

## 2.6 RNEA との違い

RNEA では角加速度$\ddot{q}_i$が既知なので、Forward Pass で各リンクのSpatial Accelerationを計算できる。その後、Backward Pass で必要なSpatial Force を親リンクへ伝達する。

一方、ABAでは$\ddot{q}_i$が未知なので、最初のForward Pass では加速度を計算できない。<br>
そこでBaskward Pass で子リンクの関節加速度を消去し、部分機構の等価慣性とバイアス力を計算する。<br>
その結果、最後のForward Passで根本から順番に関節角加速度を求められる。

## 3. ABAの数学的導出



ABAのPass 2 : Backward Pass では、末端リンクから親リンクに向かって関節角加速度を順次消去する処理を行っている。

### 3.1 末端リンクの関節角加速度の消去

まず末端リンク$n$では子リンクが存在しないため、1.2.1で示したリンク単体のSpatial Newton-Euler方程式より、以下の式となる。

$$
\boldsymbol{f}_n^{\text{net}} = \mathcal{I}_n \boldsymbol{A}_n + \boldsymbol{p}_n, \quad  \boldsymbol{p}_n = \text{crf}(\boldsymbol{V}_n)\mathcal{I}_n \boldsymbol{V}_n
$$

後の計算に合わせるために、1.2.2で示した式の形式で末端リンクの運動を表す。

$$
\boldsymbol{f}_n^{A} = \boldsymbol{I}_n^A \boldsymbol{A}_n + \boldsymbol{p}_n^A  \tag{3.1}
$$

$\boldsymbol{A}_n$ は 2.3.2節 で示したSpatial Accelerationの再帰式より、末端リンクの加速度 $\boldsymbol{A}_n$は、親リンクn-1の加速度$\boldsymbol{A}_{n-1}$と関節角加速度$\ddot{q}_n$を用いて表される。Backward Pass ではこれらを未知数として扱い、末端リンク$n$における関節角加速度$\ddot{q}_n$を代数的に消去する。

$$
\boldsymbol{A}_n = X_n \boldsymbol{A}_{n-1} + \boldsymbol{S}_n \ddot{q}_n +\boldsymbol{c}_n, \quad \boldsymbol{c}_n = \text{crm}(\boldsymbol{V}_n) \boldsymbol{S}_n \dot{q}_n  \tag{3.2}
$$

(3.2)式の$\boldsymbol{A}_n$ を (3.1)式に代入すると以下となる。

$$
\boldsymbol{f}_n^{A} = \boldsymbol{I}_n^A X_n \boldsymbol{A}_{n-1} + \boldsymbol{I}_n^A \boldsymbol{S}_n \ddot{q}_n + \boldsymbol{I}_n^A\boldsymbol{c}_n + \boldsymbol{p}_n^A \tag{3.3}
$$

[RNEAのNote](./Spatial-RNEA.ipynb)の1節 "RNEAとSubspaceとの関連"で求めたように、Spatial Force Vector から関節トルクを取り出すには、仕事率の関係より、Subspace を以下のように掛ければよい。Pass 2 : Backward Pass では関節トルクは既知の値であるが、この関係式は成立する。

$$
\tau_n = \boldsymbol{S}_n^T \boldsymbol{f}_n^{A}
$$

(3.3)式の両辺の左側から $\boldsymbol{S}_n^T$ を掛けると以下となる。

$$
\boldsymbol{S}_n^T \boldsymbol{f}_n^{A} = \boldsymbol{S}_n^T \boldsymbol{I}_n^A X_n \boldsymbol{A}_{n-1} + \boldsymbol{S}_n^T \boldsymbol{I}_n^A \boldsymbol{S}_n \ddot{q}_n + \boldsymbol{S}_n^T \boldsymbol{I}_n^A\boldsymbol{c}_n + \boldsymbol{S}_n^T \boldsymbol{p}_n^A  \tag{3.4}
$$

式の整理のため、以下を定義する。

$$
\boldsymbol{U}_n = \boldsymbol{I}_n^A \boldsymbol{S}_n  , \quad d_n = \boldsymbol{S}_n^T \boldsymbol{I}_n^A \boldsymbol{S}_n = \boldsymbol{S}_n^T \boldsymbol{U}_n 
$$

末端リンクでは $\boldsymbol{I}_n^A = \mathcal{I}_n$である。このSpatial Inertia は[このNote](spatial_vector.ipynb)の4.2節で導出した以下の末端リンクの基準点$O_n$の式であり、これは対称行列である。

$$
\mathcal{I}_{n,O} = 
\begin{bmatrix}
\boldsymbol{I}_{n,G_n} - m_n [\boldsymbol{r}_{O_n G_n}]^\land [\boldsymbol{r}_{O_n G_n}]^\land & m_n [\boldsymbol{r}_{O_n G_n}]^\land \\
- m_n [\boldsymbol{r}_{O_n G_n}]^\land & m_n \boldsymbol{I}_3 
\end{bmatrix} \in \mathbf{R}^{6 \times 6}
$$

よって以下となる。

$$
\boldsymbol{U}_n^T = \boldsymbol{S}_n^T \boldsymbol{I}_n^A 
$$

$\boldsymbol{U}_n^T, d_n$ を用いて(3.4)式を整理すると以下となる。

$$
\tau_n = \boldsymbol{U}_n^T X_n \boldsymbol{A}_{n-1} + d_n \ddot{q}_n + \boldsymbol{U}_n^T \boldsymbol{c}_n + \boldsymbol{S}_n^T \boldsymbol{p}_n^A  \tag{3.5}
$$

(3.5)式を$\ddot{q}_n$について整理すると、以下となる。

$$
\ddot{q}_n = d_n^{-1} \left(\tau_n - \boldsymbol{S}_n^T \boldsymbol{p}_n^A - \boldsymbol{U}_n^T ( X_n \boldsymbol{A}_{n-1} + \boldsymbol{c}_n) \right)
$$

関節トルク$\tau_n$は既知であり、バイアス力$\boldsymbol{p}_n^A$も既知の状態量と外力から計算できる。そこで、関節トルクからバイアス力の関節方向成分を差し引いた量を$u_n$として表す。

$$
u_n = \tau_n - \boldsymbol{S}_n^T \boldsymbol{p}_n^A 
$$

これより、角加速度の式は以下となる。Pass 3 ではこの式を用いて角加速度を求める。そのために、Pass で $\boldsymbol{c}_n$ 、Pass 2で$d_n, u_n, \boldsymbol{U}_n$ の計算を行っている。

$$
\ddot{q}_n = {d_n}^{-1} \left( u_n - \boldsymbol{U}_n^T ( X_n \boldsymbol{A}_{n-1} + \boldsymbol{c}_n) \right) \tag{3.6}
$$

$\ddot{q}_n$ を (3.3)式に代入して角加速度を消去する。

$$
\begin{aligned}
\boldsymbol{f}_n^{A} &= \boldsymbol{I}_n^A X_n \boldsymbol{A}_{n-1} + \boldsymbol{I}_n^A \boldsymbol{S}_n {d_n}^{-1} \left( u_n - \boldsymbol{U}_n^T ( X_n \boldsymbol{A}_{n-1} + \boldsymbol{c}_n) \right) + \boldsymbol{I}_n^A\boldsymbol{c}_n + \boldsymbol{p}_n^A \\
&= \boldsymbol{I}_n^A X_n \boldsymbol{A}_{n-1} + \boldsymbol{I}_n^A\boldsymbol{S}_n {d_n}^{-1} u_n - \boldsymbol{I}_n^A\boldsymbol{S}_n {d_n}^{-1} \boldsymbol{U}_n^T X_n \boldsymbol{A}_{n-1} - \boldsymbol{I}_n^A\boldsymbol{S}_n {d_n}^{-1} \boldsymbol{U}_n^T \boldsymbol{c}_n + \boldsymbol{I}_n^A \boldsymbol{c}_n + \boldsymbol{p}_n^A
\end{aligned}
$$

$\boldsymbol{I}_n^A \boldsymbol{S}_n =  \boldsymbol{U}_n$ より、

$$
\begin{aligned}
\boldsymbol{f}_n^A 
&= \boldsymbol{I}_n^A X_n \boldsymbol{A}_{n-1} - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T X_n \boldsymbol{A}_{n-1} + \boldsymbol{p}_n^A  + \boldsymbol{I}_n^A \boldsymbol{c}_n - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T \boldsymbol{c}_n + \boldsymbol{U}_n {d_n}^{-1} u_n \\
&= \left( \boldsymbol{I}_n^A - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T \right) X_n \boldsymbol{A}_{n-1} + \boldsymbol{p}_n^A  + \left(\boldsymbol{I}_n^A - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T \right) \boldsymbol{c}_n + \boldsymbol{U}_n {d_n}^{-1} u_n
\end{aligned} \tag{3.7}
$$

上式の( )の個所を以下で定義する。この式の$\boldsymbol{I}_n^A$がABIで、$\bar{\boldsymbol{I}}_n^A$が親リンクへ伝える子リンクの等価慣性である。

$$
\bar{\boldsymbol{I}}_n^A = \boldsymbol{I}_n^A - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T
$$

------------------------------------------------------------------------------------

この等価慣性の式は、Schur補完（Schur complement）に相当する。Schur補完とは、ブロック行列で表された連立方程式から一部の未知変数を消去した際に現れる係数行列である。

今回は、関節角加速度 $\ddot q_n$ を代数的に消去することで、残る親リンクの加速度に対する等価慣性が上式となった。これはSchur補完の形であり、関節自由度を考慮した等価慣性を表している。

------------------------------------------------------------------------------------

バイアス力を以下のように定義する。これは親リンクへ伝達するバイアス力である。

$$
\bar{\boldsymbol{p}}_n^A = \boldsymbol{p}_n^A  + \bar{\boldsymbol{I}}_n^A \boldsymbol{c}_n + \boldsymbol{U}_n {d_n}^{-1} u_n
$$

とすると、(3.7)式は等価慣性を用いて以下で記述できる。

$$
\boldsymbol{f}_n^A = \bar{\boldsymbol{I}}_n^A X_n \boldsymbol{A}_{n-1} + \bar{\boldsymbol{p}}_n^A \tag{3.8}
$$

この(3.8)式では、末端リンクの関節角加速度$\ddot{q}_n$が消去されており、末端リンクのSpatial Force を、未知の親リンクの$\boldsymbol{A}_{n-1}$のみを用いて表すことが出来た。<br>
ここでは$\boldsymbol{A}_{n-1}$や$\ddot{q}_n$の数値を求めていない。関節角加速度を代数的に消去しただけである。実際の関節角加速度はPass 3 で(3.6)式を用いて計算を行う。

### 3.2 親リンクへの伝達

末端リンクnの親リンクn-1単体の運動方程式は以下である。

$$
\boldsymbol{f}_{n-1}^{\text{net}} = \mathcal{I}_{n-1} \boldsymbol{A}_{n-1} + \boldsymbol{p}_{n-1} \tag{3.9}
$$

これに末端リンク$n$で計算した、末端リンクの角加速度$\ddot{q}_n$が消去された式(3.8)のSpatial Forceを親リンクへ伝達する。そこで、(3.8)式を親リンク座標へ変換する。

$$
{X_n}^T \boldsymbol{f}_n^A = {X_n}^T \bar{\boldsymbol{I}}_n^A X_n \boldsymbol{A}_{n-1} + {X_n}^T \bar{\boldsymbol{p}}_n^A \tag{3.10}
$$

(3.9)式に(3.10)式のSpatial Force を加える。これにより、親リンクの単体リンクだけでなく、子リンクも考慮した運動方程式となる。

$$
\begin{aligned}
\boldsymbol{f}_{n-1}^{A} &= \mathcal{I}_{n-1} \boldsymbol{A}_{n-1} + \boldsymbol{p}_{n-1} + {X_n}^T \boldsymbol{f}_n^A \\
&= \mathcal{I}_{n-1} \boldsymbol{A}_{n-1} + \boldsymbol{p}_{n-1} + {X_n}^T \bar{\boldsymbol{I}}_n^A X_n \boldsymbol{A}_{n-1} + {X_n}^T \bar{\boldsymbol{p}}_n^A \\
&= \left(\mathcal{I}_{n-1} + {X_n}^T \bar{\boldsymbol{I}}_n^A X_n \right) \boldsymbol{A}_{n-1} + \boldsymbol{p}_{n-1} + {X_n}^T \bar{\boldsymbol{p}}_n^A
\end{aligned}
$$

ここで、次のように定義することで、子リンクの部分機構を考慮する形になる。

$$
\boldsymbol{I}_{n-1}^A = \mathcal{I}_{n-1} + {X_n}^T \bar{\boldsymbol{I}}_n^A X_n 
$$

$$
\boldsymbol{p}_{n-1}^A = \boldsymbol{p}_{n-1} + {X_n}^T \bar{\boldsymbol{p}}_n^A
$$

これより、親リンクn-1における子リンクの部分機構を考慮した運動方程式を以下とすることが出来る。$\boldsymbol{I}_{n-1}^A$は親リンクn-1のABIである。

$$
\boldsymbol{f}_{n-1}^{A} = \boldsymbol{I}_{n-1}^A \boldsymbol{A}_{n-1} + \boldsymbol{p}_{n-1}^A \tag{3.11}
$$

この(3.11)式を用いて3.1節と同様に計算を進めていくとリンクn-1における角加速度の計算式((3.6)式)と、角加速度を消去したSpatial Forceの式((3.8)式)が現れる。そしてそのSpatial Forceをその親リンクへ伝達させ、この節のように運動方程式((3.11)式)を構築する、という流れになる。これがBackward Passである。

### 3.3 ABIの対称性について

末端リンクとの違いは$\boldsymbol{I}_{n-1}^A$に現れる。末端リンク$\boldsymbol{I}_{n}^A$は、3.1節に示したように対称行列である。親リンクのABI$\boldsymbol{I}_{n-1}^A$はリンク単体のSpatial Inertia $\mathcal{I}_{n-1}$ と子リンクの等価慣性を座標変換した項の和である。以下ではこのABIも対称行列となることを示す。

等価慣性は以下である。

$$
\bar{\boldsymbol{I}}_n^A = \boldsymbol{I}_n^A - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T
$$

この転置は$d_n$がスカラーであるため、$\bar{\boldsymbol{I}}_n^A$は対称行列である。

$$
(\bar{\boldsymbol{I}}_n^A)^T = {\boldsymbol{I}_n^A}^T - (\boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T)^T = {\boldsymbol{I}_n^A} - \boldsymbol{U}_n {d_n}^{-1} \boldsymbol{U}_n^T = \bar{\boldsymbol{I}}_n^A 
$$

そして、$\boldsymbol{I}_{n-1}^A$ を転置すると、以下となるため $\boldsymbol{I}_{n-1}^A$は対称行列である。

$$
(\boldsymbol{I}_{n-1}^A)^T = {\mathcal{I}_{n-1}}^T + ({X_n}^T \bar{\boldsymbol{I}}_n^A X_n)^T = \mathcal{I}_{n-1} + {X_n}^T (\bar{\boldsymbol{I}}_n^A)^T X_n =  \mathcal{I}_{n-1} + {X_n}^T \bar{\boldsymbol{I}}_n^A X_n = \boldsymbol{I}_{n-1}^A
$$

よって、中間変数の$\boldsymbol{U}_{n-1}$の転置を以下のように計算できる。

$$
\boldsymbol{U}_{n-1} = \boldsymbol{I}_{n-1}^A \boldsymbol{S}_{n-1} \rightarrow {\boldsymbol{U}_{n-1}}^T = {\boldsymbol{S}_{n-1}}^T \boldsymbol{I}_{n-1}^A 
$$

以上より、ABIの対称性はBackward Passの再帰計算において維持され、各リンクで3.1節と同様の関節角加速度の消去が可能となる。これにより、Backward Passの計算が成立することを確認した。実際の関節角加速度は、2.5節に示したForward Passによって求める。

以上